We will implement here a MLP (MultiLayer Perceptron) neural network trying to improve the metrics evaluated on ML best models XGBoost.

The MLP model sued will be uploaded from sklearn.

In [34]:
import yfinance as yf
import numpy as np
import pandas as pd
import warnings
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, balanced_accuracy_score, f1_score, confusion_matrix
from sklearn.preprocessing import StandardScaler
from statsmodels.tsa.arima.model import ARIMA
from sklearn.neural_network import MLPClassifier
from statsmodels.tools.sm_exceptions import ConvergenceWarning as StatsmodelsConvergenceWarning
from statsmodels.tools.sm_exceptions import ValueWarning
from sklearn.exceptions import ConvergenceWarning as SklearnConvergenceWarning
from tqdm import tqdm
import datetime

tickers = ['NVDA', 'GOOGL', 'AAPL', 'GOOG', 'MSFT', 'AMZN', 'TSM', 'AVGO', 'META', 'TSLA', 'SMCI', 'ACN']

forex_tickers = [
    "EURUSD=X",
    "GBPUSD=X",
    "USDJPY=X",
    "USDCHF=X",
    "AUDUSD=X",
    "USDCAD=X",
    "NZDUSD=X",
    "EURGBP=X",
    "EURJPY=X",
    "GBPJPY=X",
    "EURCHF=X"
]

# for log and output save
now = datetime.datetime.now()
date = now.strftime("%Y-%m-%d %H:%M:%S")

warnings.filterwarnings("ignore", category=SklearnConvergenceWarning)
warnings.filterwarnings("ignore", category=StatsmodelsConvergenceWarning)

In [35]:
def load_stock(start_date, stock='NVDA'):
    df = yf.download(stock, start=start_date, end='2026-01-01', interval='1d')

    if isinstance(df.columns, pd.MultiIndex):
        lvl0 = df.columns.get_level_values(0)
        lvl1 = df.columns.get_level_values(1)

        if 'Close' in lvl0:
            df.columns = lvl0
        elif 'Close' in lvl1:
            df.columns = lvl1

    df = df.loc[:, ~df.columns.duplicated()].copy()
    df.index = pd.to_datetime(df.index)
    df = df.sort_index()

    for col in ['Open', 'High', 'Low', 'Close', 'Volume']:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors='coerce')

    df = df.dropna(subset=['Close']).copy()
        
    return df

In [36]:
data = load_stock(start_date='2020-01-01',stock='ACN')

[*********************100%***********************]  1 of 1 completed


In [41]:
# ARIMA prediction function

def get_arima_feature(data, input=(4,0,1), window=252):
    df = data.copy()

    df['arima_pred'] = np.nan
    y = df['ret_1']

    for i in tqdm(range(1, len(df))):
        start_idx = max(0, i - window)
        history = y.iloc[start_idx:i].dropna()

        if len(history) < 10:
            continue

        try:
            # turn down warnings
            with warnings.catch_warnings():
                warnings.simplefilter("ignore", StatsmodelsConvergenceWarning)
                warnings.simplefilter("ignore", ValueWarning)
                warnings.simplefilter("ignore", FutureWarning)

                model = ARIMA(history, order=input)
                fit = model.fit()
                pred = fit.forecast(steps=1).iloc[0]
                
            df.iloc[i, df.columns.get_loc('arima_pred')] = pred

        except Exception:
            continue

    df['arima_sgn'] = np.sign(df['arima_pred']).fillna(0).astype(int)
    df['arima_intensity'] = df['arima_pred'].abs()

    return df

In [42]:
# new feature selected by correlation

def build_feature(data, nxxs=3, nxs=5, ns=10, nm=20, nb=50, dropna=True):

    data = data.copy()

    ### returns ###
    data['ret_1'] = data['Close'].pct_change(1) # (data['Close]-data['Close].shift(1))/data['Close].shift(1) -1
    data['ret_5'] = data['Close'].pct_change(5)
    data['ret_10'] = data['Close'].pct_change(10)
    
    ### moving averages ###
    data[f'MA{ns}'] = data['Close'].rolling(ns).mean()
    data[f'MA{nm}'] = data['Close'].rolling(nm).mean()
    data[f'MA{nb}'] = data['Close'].rolling(nb).mean()

    ### distance from moving average ###
    # better to calculate the distance (NORMALIZED) from the MA
    data[f'MA{ns}dis'] = (data['Close']/data[f'MA{ns}'])-1
    data[f'MA{nm}dis'] = (data['Close']/data[f'MA{nm}'])-1
    data[f'MA{nb}dis'] = (data['Close']/data[f'MA{nb}'])-1

    ### exponential moving avg ###
    def multiplier(n):
        return 2 / (n + 1)

    def ema(data, n):
        col = f'EMA{n}'
        alpha = multiplier(n)

        data[col] = np.nan
        data.loc[data.index[0], col] = data['Close'].iloc[0]

        for i in range(1, len(data)):
            data.loc[data.index[i], col] = (
                alpha * data['Close'].iloc[i]
                + (1 - alpha) * data.loc[data.index[i - 1], col]
            )

        return data

    data = ema(data,ns)
    data = ema(data,nm)
    data = ema(data,nb)

    ### distance from EMA ###
    data[f'EMA{ns}dis'] = data['Close']/data[f'EMA{ns}'] - 1
    data[f'EMA{nm}dis'] = data['Close']/data[f'EMA{nm}'] - 1
    data[f'EMA{nb}dis'] = data['Close']/data[f'EMA{nb}'] - 1

    ### z-score ###
    data[f'z{nxxs}'] = (data['Close']-data['Close'].rolling(nxxs).mean())/data['Close'].rolling(nxxs).std()
    data[f'z{nxs}'] = (data['Close']-data['Close'].rolling(nxs).mean())/data['Close'].rolling(nxs).std()
    data[f'z{ns}'] = (data['Close']-data['Close'].rolling(ns).mean())/data['Close'].rolling(ns).std()

    ### volatility ###
    data[f'vol{nxs}'] = data['ret_1'].rolling(nxs).std()
    data[f'vol{ns}'] = data['ret_1'].rolling(ns).std()
    data[f'vol{nm}'] = data['ret_1'].rolling(nm).std()

    ### bollinger bands ###
    # 2std aroud the moving avg, better to calculate the distance from the boundaries
    data[f'bb_upper_{nm}_dis'] = data['Close']/(data[f'MA{nm}'] + 2*data[f'vol{nm}']) - 1
    data[f'bb_lower_{nm}_dis'] = data['Close']/(data[f'MA{nm}'] - 2*data[f'vol{nm}']) - 1

    ### momentum ###
    data[f'momentum{ns}'] = data['Close'] - data['Close'].shift(ns)
    data[f'momentum{nm}'] = data['Close'] - data['Close'].shift(nm)

    ### ARIMA sgn and intensity ###
    data = get_arima_feature(data)

    ### tagrte ###
    data['target'] = (data['ret_1'].shift(-1) > 0).astype(int)

    ### feature list ###
    features = ['bb_lower_20_dis','MA20dis','MA50dis', 'z5' , 'EMA20dis','EMA50dis','MA10dis','EMA10dis','z3','ret_1', 'arima_sgn', 'arima_intensity']

    if dropna:
        data = data.dropna()

    return data, features

In [43]:
# define train and test split

# train_size = round(len(data) * 0.7)

split_date = '2024-06-30'

data, features = build_feature(data)

train = data.loc[data.index < split_date]
test = data.loc[data.index > split_date]

train = train.copy()
test = test.copy()

100%|██████████| 1458/1458 [00:41<00:00, 34.91it/s]


In [ ]:
# model FINE TUNE OF PARAMTERS to be tested

hidden_layer_sizes_l = [
    (16,),
    (32,),
    (64,),
    (32, 16),
    (64, 32),
    (64, 32, 16)
]
activation_l = ['relu', 'tanh']
alpha_l = [0.0001, 0.001, 0.01, 0.1]
learning_rate_l = ['constant', 'adaptive']
learning_rate_init_l = [0.0005, 0.001, 0.005]
max_iter_l = [300, 500, 1000]
batch_size_l = [32, 64, 128]

total = (
    len(hidden_layer_sizes_l) *
    len(activation_l) *
    len(alpha_l) *
    len(learning_rate_l) *
    len(learning_rate_init_l) *
    len(max_iter_l) *
    len(batch_size_l)
)

def compute_sharpe_from_signals(data, price_col='Close', signal_col='signal', annualization=252):
    df = data.copy()

    df['market_return'] = df[price_col].pct_change()
    df['strategy_return'] = df[signal_col].shift(1) * df['market_return']
    df['strategy_return'] = df['strategy_return'].fillna(0)

    returns = df['strategy_return']

    if len(returns) == 0:
        return np.nan

    std = returns.std()

    if std == 0:
        return np.nan

    sharpe = (returns.mean() / std) * np.sqrt(annualization)
    return sharpe

In [14]:
# MODEL IMPLEMENTATION

scaler = StandardScaler()
train_scaled = scaler.fit_transform(train[features])
test_scaled = scaler.transform(test[features])

results = []

pbar = tqdm(total=total)

for hidden_layer_sizes in hidden_layer_sizes_l:
    for activation in activation_l:
        for alpha in alpha_l:
            for learning_rate in learning_rate_l:
                for learning_rate_init in learning_rate_init_l:
                    for max_iter in max_iter_l:
                        for batch_size in batch_size_l:

                            model = MLPClassifier(
                                hidden_layer_sizes=hidden_layer_sizes,
                                activation=activation,
                                alpha=alpha,
                                learning_rate=learning_rate,
                                learning_rate_init=learning_rate_init,
                                max_iter=max_iter,
                                batch_size=batch_size,
                                solver='adam',
                                random_state=42
                            )

                            try:
                                model.fit(train_scaled, train['target'])
                                pred = model.predict(test_scaled)
                                proba = model.predict_proba(test_scaled)[:,1]
                            except:
                                continue
                            
                            mask = (proba > 0.6) | (proba < 0.4)

                            if mask.sum() < 20:
                                continue
                            
                            filtered_pred = pred[mask]
                            filtered_target = test['target'].iloc[mask]

                            acc = accuracy_score(filtered_target, filtered_pred)
                                
                            results.append({
                                'hidden_layer_sizes': hidden_layer_sizes,
                                'activation': activation,
                                'alpha': alpha,
                                'learning_rate': learning_rate,
                                'learning_rate_init': learning_rate_init,
                                'max_iter': max_iter,
                                'batch_size': batch_size,
                                'accuracy': acc,
                                'n_trades': mask.sum()
                            })
                            
                            if len(results) > 0:
                                best_acc = max(r['accuracy'] for r in results)
                                pbar.set_postfix({'best_acc': round(best_acc, 3)})

                            pbar.update(1)

pbar.close()

results_df = pd.DataFrame(results)

100%|█████████▉| 2580/2592 [52:28<00:14,  1.22s/it, best_acc=0.645]


In [15]:
results_df.to_excel("output/MLP_best_param.xlsx", index=False)

In [ ]:
# try the model over different stock with the best possible paramters

# model with best parameters
model = MLPClassifier(
    hidden_layer_sizes=(64,),
    activation='relu',
    alpha=0.1,
    learning_rate='adaptive',
    learning_rate_init=0.001,
    max_iter=1000,
    batch_size=32,
    solver='adam',
    random_state=42
)

scaler = StandardScaler()

train_scaled = scaler.fit_transform(train[features])
model.fit(train_scaled, train['target'])

res = []

for ticker in tickers:
    data = load_stock(start_date='2020-01-01',stock=ticker)

    data, features = build_feature(data)
    
    train = data.loc[data.index < split_date]
    test = data.loc[data.index > split_date]

    train_scaled = scaler.fit_transform(train[features])
    test_scaled = scaler.transform(test[features])

    pred = model.predict(test_scaled)
    proba = model.predict_proba(test_scaled)[:,1]

    mask = (proba > 0.6) | (proba < 0.4)

    filtered_pred = pred[mask]
    filtered_target = test['target'].iloc[mask]

    acc = accuracy_score(filtered_target, filtered_pred)

    res.append({
        'ticker': ticker,
        'acc': acc
    })

[*********************100%***********************]  1 of 1 completed
100%|██████████| 1507/1507 [00:53<00:00, 27.96it/s]
[*********************100%***********************]  1 of 1 completed
  0%|          | 0/1507 [00:00<?, ?it/s]/Users/Speranza/Desktop/VSCODE/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
100%|██████████| 1507/1507 [00:44<00:00, 34.10it/s]
[*********************100%***********************]  1 of 1 completed
  0%|          | 0/1507 [00:00<?, ?it/s]/Users/Speranza/Desktop/VSCODE/.venv/lib/python3.11/site-packages/statsmodels/tsa/statespace/sarimax.py:966: UserWarning: Non-stationary starting autoregressive parameters found. Using zeros as starting parameters.
  warn('Non-stationary starting autoregressive parameters'
/Users/Speranza/Desktop/VSCODE/.venv/lib/python3.11/site-pack

In [47]:
df = pd.DataFrame(res)
df

,ticker,acc
0,NVDA,0.495050
1,GOOGL,0.620000
2,AAPL,0.555556
3,GOOG,0.589474
4,MSFT,0.413793
5,AMZN,0.656250
6,TSM,0.575221
7,AVGO,0.522293
8,META,0.528090
9,TSLA,0.452830
